In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
import pandas as pd

val_xgb = pd.read_csv('/kaggle/input/datasets/annakorzhenko/predictions/xgb_oof_predictions.csv')
val_dl2 = pd.read_csv('/kaggle/input/datasets/annakorzhenko/predictions/dl2_oof_predictions.csv')

# merge on id
val_df = val_xgb.merge(val_dl2[['id', 'oof_dl2']], on='id')

# fit the new learer, type spec ridge
features = ['oof_xgb', 'oof_dl2']
val_df['ensemble_pred'] = 0.0
meta_models = {}

for t in val_df['type'].unique():
    idx = val_df['type'] == t
    X_type = val_df.loc[idx, features]
    y_type = val_df.loc[idx, 'scalar_coupling_constant']
    
    # separate ridge for coupling type
    model = Ridge(alpha=1.0)
    model.fit(X_type, y_type)
    meta_models[t] = model
    
    # ensembles pred
    val_df.loc[idx, 'ensemble_pred'] = model.predict(X_type)

def comp_log_mae(y_true, y_pred, types):
    df = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred, 'type': types})
    scores = []
    for t, group in df.groupby('type'):
        mae = np.mean(np.abs(group['y_true'] - group['y_pred']))
        scores.append(np.log(max(mae, 1e-9)))
    return np.mean(scores)

score_xgb = comp_log_mae(val_df['scalar_coupling_constant'], val_df['oof_xgb'], val_df['type'])
score_dl2 = comp_log_mae(val_df['scalar_coupling_constant'], val_df['oof_dl2'], val_df['type'])
score_ens = comp_log_mae(val_df['scalar_coupling_constant'], val_df['ensemble_pred'], val_df['type'])

print("Local validation (Log-MAE")
print(f"XGBoost:   {score_xgb:.4f}")
print(f"DL Model 2: {score_dl2:.4f}")
print(f"2-Model Ensemble: {score_ens:.4f}")

Local validation (Log-MAE
XGBoost:   0.6939
DL Model 2: 1.1381
2-Model Ensemble: 0.6867


In [4]:
import pandas as pd
DATA_DIR = '/kaggle/input/competitions/champs-scalar-coupling'
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')

# test predictions for both models
sub_xgb = pd.read_csv('/kaggle/input/datasets/annakorzhenko/submisisons/submission.csv')
sub_dl2 = pd.read_csv('/kaggle/input/datasets/annakorzhenko/submisisons/submission_dl2.csv')

# merge into the test df
ensemble_test_df = test_df[['id', 'type']].copy()
ensemble_test_df = ensemble_test_df.merge(sub_xgb.rename(columns={'scalar_coupling_constant': 'oof_xgb'}), on='id')
ensemble_test_df = ensemble_test_df.merge(sub_dl2.rename(columns={'scalar_coupling_constant': 'oof_dl2'}), on='id')

# target column
ensemble_test_df['scalar_coupling_constant'] = 0.0
features = ['oof_xgb', 'oof_dl2']

# type specific ridge models
for t, model in meta_models.items():
    idx = ensemble_test_df['type'] == t
    X_test_type = ensemble_test_df.loc[idx, features]
    # Predict the final test values
    ensemble_test_df.loc[idx, 'scalar_coupling_constant'] = model.predict(X_test_type)

submission_ensemble = ensemble_test_df[['id', 'scalar_coupling_constant']]
submission_ensemble.to_csv('submission_ensemble_final.csv', index=False)
print("Saved submission_ensemble_final.csv")

Saved submission_ensemble_final.csv


In [3]:
import pandas as pd

sub_xgb = pd.read_csv('/kaggle/input/datasets/annakorzhenko/submisisons/submission.csv')
sub_dl2 = pd.read_csv('/kaggle/input/datasets/annakorzhenko/submisisons/submission_dl2.csv')

print(f"XGB rows: {len(sub_xgb):,} | Unique IDs: {sub_xgb['id'].nunique():,}")
print(f"DL2 rows: {len(sub_dl2):,} | Unique IDs: {sub_dl2['id'].nunique():,}")

XGB rows: 2,505,190 | Unique IDs: 2,505,190
DL2 rows: 2,505,190 | Unique IDs: 2,505,190


In [5]:
!kaggle competitions submit -c champs-scalar-coupling -f submission_ensemble_final.csv -m "2 models ensemble, XGBoost + two-branch DL"

100%|██████████████████████████████████████| 63.8M/63.8M [00:00<00:00, 80.6MB/s]
Successfully submitted to Predicting Molecular Properties